# Download SOOP Dataset (OpenNeuro ds004889)

This notebook downloads the SOOP stroke dataset and saves it as Kaggle output.

**After running:**
1. Click "Save Version" -> "Save & Run All" -> "Save"
2. Go to the Output tab of this notebook
3. In your training notebook, click "Add Input" -> "Your Work" -> select this notebook's output

**Requirements:**
- Internet: ON
- Accelerator: None (GPU not needed)
- Runtime: ~20-30 min

In [ ]:
!pip install -q awscli

In [ ]:
import os
from pathlib import Path

SOOP_ROOT = Path("/kaggle/working/ds004889")
SOOP_ROOT.mkdir(parents=True, exist_ok=True)

# Check available disk space
!df -h /kaggle/working

In [ ]:
# Step 1: Download participants.tsv first
print("Downloading participants.tsv...")
!aws s3 cp --no-sign-request \
    s3://openneuro.org/ds004889/participants.tsv \
    {SOOP_ROOT}/participants.tsv

# Check it
import csv
with open(SOOP_ROOT / "participants.tsv") as f:
    reader = csv.DictReader(f, delimiter="\t")
    rows = list(reader)
print(f"Total participants: {len(rows)}")
print(f"Columns: {list(rows[0].keys())}")

# Count confirmed stroke
stroke_count = sum(1 for r in rows if r.get("acuteischaemicstroke", "").lower() == "yes")
print(f"Confirmed acute ischemic stroke: {stroke_count}")

In [ ]:
# Step 2: Download DWI + ADC (dwi/ folders, .nii.gz only)
print("Downloading DWI + ADC files...")
print("(This takes ~10-15 min)")
!aws s3 sync --no-sign-request \
    s3://openneuro.org/ds004889/ {SOOP_ROOT}/ \
    --exclude "*" \
    --include "sub-*/dwi/*.nii.gz" \
    --no-progress

print("\nDWI+ADC download complete!")
!du -sh {SOOP_ROOT}

In [ ]:
# Step 3: Download FLAIR only (skip T1w to save space)
print("Downloading FLAIR files...")
print("(This takes ~5-10 min)")
!aws s3 sync --no-sign-request \
    s3://openneuro.org/ds004889/ {SOOP_ROOT}/ \
    --exclude "*" \
    --include "sub-*/anat/*FLAIR*.nii.gz" \
    --no-progress

print("\nFLAIR download complete!")
!du -sh {SOOP_ROOT}

In [ ]:
# Step 4: Download lesion masks
print("Downloading lesion masks...")
!aws s3 sync --no-sign-request \
    s3://openneuro.org/ds004889/ {SOOP_ROOT}/ \
    --exclude "*" \
    --include "derivatives/lesion_masks/*.nii.gz" \
    --include "derivatives/lesion_masks/*/*.nii.gz" \
    --include "derivatives/lesion_masks/*/*/*.nii.gz" \
    --include "derivatives/lesion_masks/*/*/*/*.nii.gz" \
    --no-progress

print("\nMasks download complete!")
!du -sh {SOOP_ROOT}

In [ ]:
# Step 5: Verify download
soop_subs = sorted([d.name for d in SOOP_ROOT.iterdir() if d.name.startswith("sub-")])
print(f"Total subject folders: {len(soop_subs)}")

# Check file counts
dwi_count = 0
adc_count = 0
flair_count = 0
mask_count = 0

for sub_id in soop_subs:
    sub_dir = SOOP_ROOT / sub_id
    dwi_dir = sub_dir / "dwi"
    anat_dir = sub_dir / "anat"
    
    if dwi_dir.exists():
        for f in dwi_dir.iterdir():
            if "TRACE" in f.name and f.suffix == ".gz":
                dwi_count += 1
            if "ADC" in f.name and f.suffix == ".gz":
                adc_count += 1
    if anat_dir.exists():
        for f in anat_dir.iterdir():
            if "FLAIR" in f.name and f.suffix == ".gz":
                flair_count += 1

# Count masks
mask_dir = SOOP_ROOT / "derivatives" / "lesion_masks"
if mask_dir.exists():
    for f in mask_dir.rglob("*lesion_mask.nii.gz"):
        mask_count += 1

print(f"\nFile counts:")
print(f"  DWI (TRACE): {dwi_count}")
print(f"  ADC:         {adc_count}")
print(f"  FLAIR:       {flair_count}")
print(f"  Masks:       {mask_count}")

# Check one subject
if soop_subs:
    test_sub = soop_subs[0]
    print(f"\nSample subject: {test_sub}")
    for root, dirs, files in os.walk(SOOP_ROOT / test_sub):
        for f in files:
            print(f"  {os.path.relpath(os.path.join(root, f), SOOP_ROOT / test_sub)}")

In [ ]:
# Final disk usage
print("Disk usage:")
!du -sh {SOOP_ROOT}
!df -h /kaggle/working

print("\n" + "="*60)
print("SOOP download complete!")
print("="*60)
print("\nNext steps:")
print('1. Click "Save Version" at top right')
print('2. Choose "Save & Run All (Commit)"')
print('3. Wait for the save to complete')
print('4. In training notebook: Add Input -> Your Work -> this notebook output')